In [7]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os
os.chdir("/content")
print(os.getcwd())


/content


In [9]:
!rm -rf /content/Task-driven-low-light-enhancement
!git clone https://github.com/SarthakBaghel/Task-driven-low-light-enhancement.git /content/Task-driven-low-light-enhancement
%cd /content/Task-driven-low-light-enhancement
!pip install -r requirements.txt


Cloning into '/content/Task-driven-low-light-enhancement'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 72 (delta 15), reused 67 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 158.34 KiB | 1.72 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Task-driven-low-light-enhancement


In [10]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: Tesla T4


In [11]:
from pathlib import Path
import os
import shutil

DRIVE_ROOT = Path("/content/drive/MyDrive")
PIPELINE_ROOT = DRIVE_ROOT / "task_driven_video_pipeline"
KAGGLE_V1_ROOT = PIPELINE_ROOT / "kaggle_v1"
CHECKPOINTS_ROOT = DRIVE_ROOT / "task_driven_checkpoints" / "kaggle_v1"

RUN_TAG = "subject_class_balanced_20k"

TRAIN_LOWLIGHT_ROOT = KAGGLE_V1_ROOT / f"train_lowlight_{RUN_TAG}_eye_mid"
VAL_LOWLIGHT_ROOT = KAGGLE_V1_ROOT / f"val_lowlight_{RUN_TAG}_eye_mid"
TEST_LOWLIGHT_ROOT = KAGGLE_V1_ROOT / f"test_lowlight_{RUN_TAG}_eye_mid"

TRAIN_CLEAN_ROOT = KAGGLE_V1_ROOT / f"train_clean_{RUN_TAG}"
VAL_CLEAN_ROOT = KAGGLE_V1_ROOT / f"val_clean_{RUN_TAG}"
TEST_CLEAN_ROOT = KAGGLE_V1_ROOT / f"test_clean_{RUN_TAG}"

FULL_CLEAN_CKPT_DIR = CHECKPOINTS_ROOT / f"full_clean_detector_{RUN_TAG}_resnet18"
FULL_CLEAN_BEST_CKPT = FULL_CLEAN_CKPT_DIR / f"kaggle_v1_clean_full_{RUN_TAG}_best.pt"

LOWLIGHT_BUNDLE_ROOT = Path(f"/content/kaggle_v1_lowlight_bundle_{RUN_TAG}_eye_mid")
LOWLIGHT_TRAIN_LINK = LOWLIGHT_BUNDLE_ROOT / "train"
LOWLIGHT_VAL_LINK = LOWLIGHT_BUNDLE_ROOT / "val"

LOWLIGHT_CKPT_DIR = CHECKPOINTS_ROOT / f"detector_lowlight_{RUN_TAG}_eye_mid_resnet18"
LOWLIGHT_BEST_CKPT = LOWLIGHT_CKPT_DIR / "detector_lowlight_best.pt"
LOWLIGHT_LAST_CKPT = LOWLIGHT_CKPT_DIR / "detector_lowlight_last.pt"

assert TRAIN_LOWLIGHT_ROOT.exists(), f"Missing: {TRAIN_LOWLIGHT_ROOT}"
assert VAL_LOWLIGHT_ROOT.exists(), f"Missing: {VAL_LOWLIGHT_ROOT}"
assert TEST_LOWLIGHT_ROOT.exists(), f"Missing: {TEST_LOWLIGHT_ROOT}"
assert FULL_CLEAN_BEST_CKPT.exists(), f"Missing: {FULL_CLEAN_BEST_CKPT}"

print("TRAIN_LOWLIGHT_ROOT:", TRAIN_LOWLIGHT_ROOT)
print("VAL_LOWLIGHT_ROOT:", VAL_LOWLIGHT_ROOT)
print("TEST_LOWLIGHT_ROOT:", TEST_LOWLIGHT_ROOT)
print("FULL_CLEAN_BEST_CKPT:", FULL_CLEAN_BEST_CKPT)
print("LOWLIGHT_BUNDLE_ROOT:", LOWLIGHT_BUNDLE_ROOT)
print("LOWLIGHT_CKPT_DIR:", LOWLIGHT_CKPT_DIR)


TRAIN_LOWLIGHT_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/train_lowlight_subject_class_balanced_20k_eye_mid
VAL_LOWLIGHT_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/val_lowlight_subject_class_balanced_20k_eye_mid
TEST_LOWLIGHT_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_lowlight_subject_class_balanced_20k_eye_mid
FULL_CLEAN_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_full_subject_class_balanced_20k_best.pt
LOWLIGHT_BUNDLE_ROOT: /content/kaggle_v1_lowlight_bundle_subject_class_balanced_20k_eye_mid
LOWLIGHT_CKPT_DIR: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/detector_lowlight_subject_class_balanced_20k_eye_mid_resnet18


In [12]:
!find "{TRAIN_LOWLIGHT_ROOT}/open" -type f | wc -l
!find "{TRAIN_LOWLIGHT_ROOT}/closed" -type f | wc -l

!find "{VAL_LOWLIGHT_ROOT}/open" -type f | wc -l
!find "{VAL_LOWLIGHT_ROOT}/closed" -type f | wc -l

!find "{TEST_LOWLIGHT_ROOT}/open" -type f | wc -l
!find "{TEST_LOWLIGHT_ROOT}/closed" -type f | wc -l


383
383
973
973
8644
8644


In [13]:
if LOWLIGHT_BUNDLE_ROOT.exists():
    shutil.rmtree(LOWLIGHT_BUNDLE_ROOT)

LOWLIGHT_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)

os.symlink(TRAIN_LOWLIGHT_ROOT, LOWLIGHT_TRAIN_LINK)
os.symlink(VAL_LOWLIGHT_ROOT, LOWLIGHT_VAL_LINK)

print("Created low-light bundle:", LOWLIGHT_BUNDLE_ROOT)
print("train ->", LOWLIGHT_TRAIN_LINK.resolve())
print("val   ->", LOWLIGHT_VAL_LINK.resolve())


Created low-light bundle: /content/kaggle_v1_lowlight_bundle_subject_class_balanced_20k_eye_mid
train -> /content/drive/.shortcut-targets-by-id/1PGJCrZrodZvhgcqSmVi73Qs7wZAAByFc/task_driven_video_pipeline/kaggle_v1/train_lowlight_subject_class_balanced_20k_eye_mid
val   -> /content/drive/.shortcut-targets-by-id/1PGJCrZrodZvhgcqSmVi73Qs7wZAAByFc/task_driven_video_pipeline/kaggle_v1/val_lowlight_subject_class_balanced_20k_eye_mid


In [14]:
!find "{LOWLIGHT_BUNDLE_ROOT}" -maxdepth 2 -type d | sed -n '1,40p'


/content/kaggle_v1_lowlight_bundle_subject_class_balanced_20k_eye_mid


In [15]:
!python3 /content/Task-driven-low-light-enhancement/train_transfer_detector.py \
  {LOWLIGHT_BUNDLE_ROOT} \
  --runtime colab \
  --colab-workspace-root /content/Task-driven-low-light-enhancement \
  --backbone resnet18 \
  --epochs 10 \
  --batch-size 64 \
  --num-workers 2 \
  --learning-rate 3e-5 \
  --weight-decay 1e-4 \
  --monitor f1 \
  --threshold-objective f1 \
  --threshold-candidates 0.3 0.4 0.5 0.6 0.7 \
  --early-stopping-patience 4 \
  --scheduler-patience 2 \
  --deterministic \
  --detector-input-mode raw \
  --init-detector-checkpoint {FULL_CLEAN_BEST_CKPT} \
  --no-pretrained \
  --save-path artifacts/kaggle_v1_lowlight/detector_lowlight_best.pt \
  --save-last-path artifacts/kaggle_v1_lowlight/detector_lowlight_last.pt \
  --drive-checkpoint-dir {LOWLIGHT_CKPT_DIR}


Adjusted initial batch size from 64 to 24 based on available GPU memory.
Runtime: colab
Workspace root: /content/Task-driven-low-light-enhancement
Training on device: cuda
Classes: {'open': 0, 'closed': 1}
Samples: train=766 | val=1946
Focal alpha: [1.0, 1.0]
Best checkpoint path: /content/Task-driven-low-light-enhancement/artifacts/kaggle_v1_lowlight/detector_lowlight_best.pt
Last checkpoint path: /content/Task-driven-low-light-enhancement/artifacts/kaggle_v1_lowlight/detector_lowlight_last.pt
Detector initialization: loaded clean-trained detector checkpoint
Drive backup directory: /content/drive/.shortcut-targets-by-id/1_ksiIpFiqPPS0965OiT3d7wai2DO7_ZK/task_driven_checkpoints/kaggle_v1/detector_lowlight_subject_class_balanced_20k_eye_mid_resnet18
                                                          
Epoch 1/10 | lr=0.000030
Train    loss: 0.0932 | accuracy: 0.8394 | precision: 0.8368 | recall: 0.8433 | f1: 0.8401 | closed_recall: 0.8433 | threshold: 0.50
Val      loss: 0.0666 | 

In [16]:
!ls "{LOWLIGHT_CKPT_DIR}"
!ls "{LOWLIGHT_BEST_CKPT}"
!ls "{LOWLIGHT_LAST_CKPT}"


detector_lowlight_best.pt  detector_lowlight_last.pt
/content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/detector_lowlight_subject_class_balanced_20k_eye_mid_resnet18/detector_lowlight_best.pt
/content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/detector_lowlight_subject_class_balanced_20k_eye_mid_resnet18/detector_lowlight_last.pt


In [18]:
import torch
import numpy as np
import pprint

best_ckpt = torch.load(LOWLIGHT_BEST_CKPT, map_location="cpu", weights_only=False)

def to_python(obj):
    if isinstance(obj, dict):
        return {k: to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_python(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(to_python(v) for v in obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    return obj

print("Checkpoint keys:")
print(sorted(best_ckpt.keys()))
print()

print("Epoch:", best_ckpt.get("epoch"))
print("Best threshold:", best_ckpt.get("best_threshold"))
print()

print("Train metrics:")
pprint.pprint(to_python(best_ckpt.get("train_metrics", {})))
print()

print("Validation metrics:")
pprint.pprint(to_python(best_ckpt.get("val_metrics", {})))


Checkpoint keys:
['best_threshold', 'class_to_idx', 'classes', 'epoch', 'lowlight_metrics', 'model_config', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'train_metrics', 'training_config', 'val_metrics']

Epoch: 1
Best threshold: 0.4

Train metrics:
{'accuracy': 0.839425587467363,
 'balanced_accuracy': 0.8394255874673628,
 'closed_recall': 0.8433420365535248,
 'confusion_matrix': [[320, 63], [60, 323]],
 'f1': 0.8400520156046813,
 'fn': 60,
 'fp': 63,
 'loss': 0.09324058967669234,
 'precision': 0.8367875647668394,
 'predicted_negative_rate': 0.4960835509138381,
 'predicted_positive_rate': 0.5039164490861618,
 'recall': 0.8433420365535248,
 'specificity': 0.835509138381201,
 'threshold': 0.5,
 'tn': 320,
 'tp': 323}

Validation metrics:
{'accuracy': 0.9064748201438849,
 'balanced_accuracy': 0.9064748201438848,
 'closed_recall': 0.9198355601233299,
 'confusion_matrix': [[869, 104], [78, 895]],
 'f1': 0.907707910750507,
 'fn': 78,
 'fp': 104,
 'loss': 0.066582596617

In [19]:
VAL_LOWLIGHT_REPORT_DIR = KAGGLE_V1_ROOT / f"eval_detector_lowlight_on_val_{RUN_TAG}_eye_mid"

!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {LOWLIGHT_BEST_CKPT} \
  {VAL_LOWLIGHT_ROOT} \
  --batch-size 64 \
  --num-workers 0 \
  --output-dir {VAL_LOWLIGHT_REPORT_DIR}


Clean: 100% 31/31 [00:11<00:00,  2.69batch/s]
Evaluating on device: cuda
Classes: {'open': 0, 'closed': 1}
Clean    loss: 0.0666 | accuracy: 0.9065 | precision: 0.8959 | recall: 0.9198 | f1: 0.9077 | closed_recall: 0.9198 | threshold: 0.40
Saved subset manifest to: /content/drive/.shortcut-targets-by-id/1PGJCrZrodZvhgcqSmVi73Qs7wZAAByFc/task_driven_video_pipeline/kaggle_v1/eval_detector_lowlight_on_val_subject_class_balanced_20k_eye_mid/subset_manifest.csv
Saved evaluation report to: /content/drive/.shortcut-targets-by-id/1PGJCrZrodZvhgcqSmVi73Qs7wZAAByFc/task_driven_video_pipeline/kaggle_v1/eval_detector_lowlight_on_val_subject_class_balanced_20k_eye_mid

Report Table
| Dataset | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Clean | 90.65% | 89.59% | 91.98% | 90.77% |

Saved compact CSV to: /content/drive/.shortcut-targets-by-id/1PGJCrZrodZvhgcqSmVi73Qs7wZAAByFc/task_driven_video_pipeline/kaggle_v1/eval_detector_lowlight_on_val_subject_class_balanced_20k_eye_mid/exp

In [20]:
!sed -n '1,200p' "{VAL_LOWLIGHT_REPORT_DIR}/evaluation_summary.txt"


Using threshold=0.40 for the closed-eye class.
Closed-eye recall on clean data: 0.9198.

In [21]:
print("Revised Phase 7 complete.")
print("LOWLIGHT_BEST_CKPT:", LOWLIGHT_BEST_CKPT)
print("LOWLIGHT_LAST_CKPT:", LOWLIGHT_LAST_CKPT)
print("FULL_CLEAN_BEST_CKPT:", FULL_CLEAN_BEST_CKPT)


Revised Phase 7 complete.
LOWLIGHT_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/detector_lowlight_subject_class_balanced_20k_eye_mid_resnet18/detector_lowlight_best.pt
LOWLIGHT_LAST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/detector_lowlight_subject_class_balanced_20k_eye_mid_resnet18/detector_lowlight_last.pt
FULL_CLEAN_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_full_subject_class_balanced_20k_best.pt
